In [ ]:
import pandas as pd
import sys
import matplotlib.pyplot as plt
import seaborn as sns

# Custom modules
sys.path.append(('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/datasets/spe-1/spe1_helper_modules/'))
from spk_feat_cluster_comp_analysis  import *

In [ ]:
cluster_pickle_dir = "/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spe1_pickles/cluster_pickles/"

### Step 1: Create dataframe with all experiment metadata and clustering results

In [ ]:
#create df for all experiments 
df_master = compile_experiment_results(cluster_pickle_dir)


In [ ]:
df_master

In [ ]:
#sort by cell id
# Create a numeric column for sorting (c1, c2, c10, etc.)
df_master['sort_idx'] = df_master['cell_id'].str.extract('(\d+)').astype(int)

# Sort by the numeric ID, then by feature name
df_master = df_master.sort_values(by=['sort_idx', 'spike_feature']).drop(columns=['sort_idx'])



### Step 2: Generate table figure from dataframe 


In [ ]:

gen_table_fig(df_master)


### Step 3: Patterns for clustering/metadata

#### A) Do metadata features correlate with clustering features

#### B) Study metadata features that do correlate

#### C) What are the experiments with the biggest difference in waveform (nRMSE) and waveform abs shape (cos sim)

In [ ]:
# 1. Higher nRMSE = More change. Lower Cos Sim = More change.
# We rank by high nRMSE and low Cos Sim simultaneously.
df_ranked = df_master.sort_values(by=['nRMSE', 'cos_sim'], ascending=[False, True])

# 2. Grab Top N (e.g., 10) and see why they are different
top_n = df_ranked.head(20)
print(top_n[['cell_id','spike_feature', 'num_clusters', 'nRMSE', 'cos_sim']])

In [ ]:
df_ranked


In [ ]:

from scipy.stats import pearsonr

def plot_significance_half_matrix(df):
    # 1. Select and Encode Columns
    analysis_cols = [
        'patch_type', 'current_type', 'cell_type', 'cortical_depth', 
        'dark_neuron', 'clear_EAP_waveform', 'num_clusters', 'nRMSE', 'cos_sim'
    ]
    df_sub = df[analysis_cols].copy()
    
    cat_feats = ['patch_type', 'current_type', 'cell_type', 'dark_neuron', 'clear_EAP_waveform']
    for col in cat_feats:
        df_sub[col] = df_sub[col].astype('category').cat.codes
    
    df_clean = df_sub.dropna()
    
    # 2. Calculate Correlation and P-values
    corr_matrix = df_clean.corr()
    n_cols = len(corr_matrix.columns)
    p_values = np.ones((n_cols, n_cols))
    
    for i in range(n_cols):
        for j in range(i + 1, n_cols):
            _, p = pearsonr(df_clean.iloc[:, i], df_clean.iloc[:, j])
            p_values[i, j] = p
            p_values[j, i] = p

    # 3. Slice the matrix to remove first row and last column
    # Rows: from 1 to end (removes patch_type from y-axis)
    # Cols: from start to second-to-last (removes cos_sim from x-axis)
    corr_sliced = corr_matrix.iloc[1:, :-1]
    p_sliced = p_values[1:, :-1]
    
    # 4. Create Mask for the remaining triangle
    # We need a new mask shaped like the sliced matrix
    mask = np.triu(np.ones_like(corr_sliced, dtype=bool), k=1)

    # 5. Setup Plot
    plt.figure(figsize=(14, 12))
    ax = sns.heatmap(
        corr_sliced, 
        mask=mask, 
        fmt="", 
        cmap='coolwarm', 
        center=0, 
        square=True,
        linewidths=.5,
        annot=False,
        cbar_kws={"label": "Pearson Correlation ($r$)"}
    )

    # 6. Manually Add Bold Stars and Values
    n_rows_new, n_cols_new = corr_sliced.shape
    for i in range(n_rows_new):
        for j in range(n_cols_new):
            if not mask[i, j]:
                r_val = corr_sliced.iloc[i, j]
                p_val = p_sliced[i, j]
                
                stars = ""
                if p_val < 0.001: stars = "***"
                elif p_val < 0.01: stars = "**"
                elif p_val < 0.05: stars = "*"
                
                ax.text(j + 0.5, i + 0.35, stars, 
                        ha='center', va='center', color='black', 
                        fontsize=14, fontweight='bold')
                
                ax.text(j + 0.5, i + 0.65, f"{r_val:.2f}", 
                        ha='center', va='center', color='black', 
                        fontsize=11)

   
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# Run it
plot_significance_half_matrix(df_master)